# Goal 1: from protein sequences to an alignment network

## Purpose

Minimal executable pipeline: accept named amino-acid sequences, score each
unordered pair, and retain selected pairs as edges in an undirected graph. Toy
sequences permit direct inspection and support software validation.

Exact local alignment uses Biopython's maintained `PairwiseAligner`
([Cock et al., 2009](https://doi.org/10.1093/bioinformatics/btp163)), following
[Smith and Waterman (1981)](https://doi.org/10.1016/0022-2836(81)90087-5).
Project code validates sequences, coordinates all-pairs scoring, and constructs
graphs.

```text
named sequences
    -> configured local aligner
    -> symmetric score matrix
    -> explicit edge rule
    -> NetworkX graph
```

Reusable modules:

- `src/protein_alignment_networks/pipeline.py`: all-pairs scoring;
- `src/protein_alignment_networks/graphs.py`: score-to-graph conversion;
- `src/protein_alignment_networks/io.py`: validation and FASTA I/O.

Notebook 02 applies this pipeline to a Pfam pilot. Notebook 03 evaluates input
collections and graph rules at larger scale.


In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from Bio import Align
from Bio.Align import substitution_matrices

from protein_alignment_networks import (
    pairwise_score_matrix,
    score_matrix_to_graph,
)

## 1. Configure exact local protein alignment

Let $x=x_1\ldots x_p$ and $y=y_1\ldots y_q$ be protein sequences.
`mode="local"` maximises alignment score over subsequences of $x$ and $y$.
BLOSUM62 supplies substitution score $a(x_i,y_j)$
([Henikoff and Henikoff, 1992](https://doi.org/10.1073/pnas.89.22.10915)).
An affine gap of length $\ell\geq1$ costs
$g_o+(\ell-1)g_e$, where $g_o$ and $g_e$ denote gap-open and gap-extension
costs; [Gotoh (1982)](https://doi.org/10.1016/0022-2836(82)90398-9) gives an
efficient recurrence.

Configuration uses BLOSUM62, $g_o=11$, and $g_e=1$ as a conventional protein
baseline. Matrix and gap costs accompany every score interpretation.


In [2]:
aligner = Align.PairwiseAligner(
    mode='local',
    substitution_matrix=substitution_matrices.load('BLOSUM62'),
    open_gap_score=-11,
    extend_gap_score=-1,
)
print(aligner.algorithm)

Gotoh local alignment algorithm


## 2. Inspect one alignment

Let $\mathcal A_{\mathrm{local}}(x,y)$ contain valid local alignments and let
$\sigma(A)$ equal substitution total minus affine-gap costs for alignment $A$.
Biopython returns
$s(x,y)=\max_{A\in\mathcal A_{\mathrm{local}}(x,y)}\sigma(A)$ through
`score()` and coordinates of an optimiser through `align()`. Cell output is
calculated directly.


In [3]:
sequence_a = 'PAWHEAE'
sequence_b = 'HEAGAWGHEE'
alignments = aligner.align(sequence_a, sequence_b)
print(f'Score: {alignments.score:g}')
print(alignments[0] if len(alignments) else 'No positive local alignment')

Score: 17
target            1 AW-HE 5
                  0 ||-|| 5
query             4 AWGHE 9



## 3. Compute every unique pair

$n$ sequences define $P=\binom n2=n(n-1)/2$ unordered non-self pairs. Matrix
$S\in\mathbb R^{n\times n}$ stores $S_{ij}=s(x_i,x_j)$ and satisfies
$S_{ij}=S_{ji}$; diagonal entries are self-alignment scores.
`pairwise_score_matrix()` computes one triangle and mirrors it. Later scripts
store rows $i<j$ in a canonical pair table for method joins.


In [4]:
sequences = {
    'protein_a': 'PAWHEAE',
    'protein_b': 'HEAGAWGHEE',
    'protein_c': 'MKTAYIAKQRQISFVKSHFSRQ',
    'protein_d': 'PAWHDQE',
}
scores = pairwise_score_matrix(sequences, aligner=aligner)
scores

protein_id,protein_a,protein_b,protein_c,protein_d
protein_id,,,,
protein_a,44.0,17.0,8.0,36.0
protein_b,17.0,62.0,8.0,19.0
protein_c,8.0,8.0,109.0,8.0
protein_d,36.0,19.0,8.0,46.0


## 4. Convert scores into a graph

For threshold $\tau$, define $G_\tau=(V,E_\tau,w)$, where $V$ contains every
sequence,
$E_\tau=\{\{i,j\}:i<j,\ S_{ij}\geq\tau\}$, and
$w(\{i,j\})=S_{ij}$. Isolated sequences remain vertices.

$\tau$ provides a software example. Raw scores vary with sequence length,
composition, substitution matrix, gap costs, and collection. Notebook 03
evaluates scalable edge rules and parameter sweeps.


In [5]:
demonstration_threshold = 20.0
graph = score_matrix_to_graph(scores, threshold=demonstration_threshold)
print(f'Nodes: {graph.number_of_nodes()}')
print(f'Edges: {graph.number_of_edges()}')
list(graph.edges(data=True))

Nodes: 4
Edges: 1


[('protein_a', 'protein_d', {'score': 36.0})]

## Outcome

Notebook verifies in-memory flow from validated sequences to score matrix and
graph. Biological evaluation requires labelled data, method-specific coverage,
recorded provenance, and sensitivity analysis; Notebooks 02 and 03 supply
these components.

## References

BibTeX records: `references/references.bib`.

- Smith and Waterman (1981), local alignment: `smith1981identification`.
- Gotoh (1982), affine gaps: `gotoh1982improved`.
- Henikoff and Henikoff (1992), BLOSUM: `henikoff1992amino`.
- Cock et al. (2009), Biopython: `cock2009biopython`.
